In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from tensorflow.keras.utils import plot_model
import cv2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import Callback
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Dropout, Flatten, Reshape, LayerNormalization, MultiHeadAttention, Conv2D, MaxPooling2D, GlobalAveragePooling2D)
from tensorflow.keras.models import Model
from tensorflow.keras.applications import NASNetMobile
from tensorflow.keras import layers

In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
from zipfile import ZipFile
file_name= '/content/gdrive/My Drive/OCT_Dataset2.zip'
with ZipFile(file_name,'r') as zip:
  zip.extractall()
  print('Successful')

In [ ]:
# Set dataset paths
data_dir = '/content/OCT_Dataset2'

In [ ]:
# Setting the height and width of the image for the model
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMAGE_SIZE = [IMG_HEIGHT, IMG_WIDTH]

# Setting the directory of our data
DME = os.listdir(os.path.join(data_dir, 'DME'))
Healthy = os.listdir(os.path.join(data_dir, 'Healthy'))

# Concatenate two data
data = np.concatenate([DME, Healthy])
assert len(data) == len(DME) + len(Healthy)

# Mapping our classes  1 == Diabetic Macular Edema and 0 == Healthy
target_x = np.full(len(DME), 1)
target_y = np.full(len(Healthy), 0)

# Label of our data
data_target = np.concatenate([target_x, target_y])
assert len(data_target) == len(target_x) + len(target_y)
assert len(data_target) == len(data)

print(len(target_x), len(target_y))


In [ ]:
import matplotlib.pyplot as plt
# Assuming you have these variables from your previous code:
num_DME = len(DME)
num_healthy = len(Healthy)

# Create labels and data for the chart
labels = ['DME', 'Healthy']
values = [num_DME, num_healthy]

In [ ]:
plt.bar(labels, values)
plt.title('Number of Images in Each Class')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.show()

In [ ]:
# Setting the height and width of the image for our model
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMAGE_SIZE = [IMG_HEIGHT, IMG_WIDTH]

In [ ]:
# Reading images from directory, resizing to IMG_HEIGHT * IMG_WIDTH * 3 and appending images to a list
X_data = []

for file in DME:
    img = cv2.imread(os.path.join(data_dir, 'DME', file))
    img2 = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    (b, g, r) = cv2.split(img2)
    img = cv2.merge([r, g, b])
    X_data.append(img)

for file in Healthy:
    img = cv2.imread(os.path.join(data_dir, 'Healthy', file))
    img2 = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    (b, g, r) = cv2.split(img2)
    img = cv2.merge([r, g, b])
    X_data.append(img)

X_data = np.array(X_data)
X_data = X_data.astype('float32')
X_data /= 255
assert len(X_data) == len(data) == len(data_target)

In [ ]:
from sklearn.model_selection import train_test_split
# Split the data into 75:15:10 ratio
X_train, X_temp, y_train, y_temp = train_test_split(X_data, data_target, test_size=0.25, random_state=50, stratify=data_target)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.4, random_state=42, stratify=y_temp)

# Data augmentation
augmented_data_gen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.2,
    zoom_range=0.1,
    brightness_range=(0.5, 1.5),
    horizontal_flip=True,
    fill_mode='nearest'
)

**Build Hybrid Inception-NeuroFusion Model**

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy

# ----------------------------
# PatchExtractor
# ----------------------------
class PatchExtractor(layers.Layer):
    def __init__(self, patch_size=(8,8), **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, x):
        p_h, p_w = self.patch_size
        B = tf.shape(x)[0]
        H, W, C = x.shape[1], x.shape[2], x.shape[3]
        pad_h = (p_h - (H % p_h)) % p_h
        pad_w = (p_w - (W % p_w)) % p_w
        if pad_h or pad_w:
            x = tf.pad(x, [[0,0],[0,pad_h],[0,pad_w],[0,0]])
        Hn, Wn = tf.shape(x)[1]//p_h, tf.shape(x)[2]//p_w
        patches = tf.reshape(x, (B, Hn, p_h, Wn, p_w, C))
        patches = tf.reduce_mean(patches, axis=[2,4])       # (B,Hn,Wn,C)
        patches = tf.reshape(patches, (B, Hn*Wn, C))        # (B,N,C)
        return patches


class LocalTokenMixer(layers.Layer):
    """Conv1D-based local mixer across tokens: cheap local token interactions."""
    def __init__(self, kernel_size=3, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
        self.dropout = dropout
    def build(self, input_shape):
        # input_shape: (B, N, C)
        self.C = int(input_shape[-1])
        # Conv1D over tokens: we keep filters=C to preserve dims
        self.conv1d = layers.Conv1D(filters=self.C, kernel_size=self.kernel_size, padding='same', groups=1)
        self.norm = layers.LayerNormalization(epsilon=1e-6)
        self.drop = layers.Dropout(self.dropout)
        super().build(input_shape)
    def call(self, x, training=False):
        # x: (B, N, C)
        y = self.norm(x)
        y = self.conv1d(y)    # (B,N,C)
        y = self.drop(y, training=training)
        return x + y          # residual

class GroupedChannelAttention(layers.Layer):
    """Split channels into G groups, compute small attention per-group over channels.
       Implemented as: per-group linear -> softmax along channel axis -> reweight.
       Input shape: (B, N, C)
    """
    def __init__(self, groups=4, reduction=4, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.groups = groups
        self.reduction = reduction
        self.dropout = dropout
    def build(self, input_shape):
        # input_shape: (B, N, C)
        _, self.N, self.C = input_shape
        if self.C is None:
            raise ValueError("Channels must be statically known for GroupedChannelAttention.")
        if self.C % self.groups != 0:
            # fallback: reduce groups to divisor of C
            # pick largest divisor <= groups
            for g in range(self.groups, 0, -1):
                if self.C % g == 0:
                    self.groups = g
                    break
        self.Cg = self.C // self.groups
        # small projection per group
        self.proj1 = layers.Dense(self.C // self.reduction, activation='relu')  # bottleneck across channel dims
        self.proj2 = layers.Dense(self.C)  # restore
        self.norm = layers.LayerNormalization(epsilon=1e-6)
        self.drop = layers.Dropout(self.dropout)
        super().build(input_shape)
    def call(self, x, training=False):
        # x: (B,N,C)
        B = tf.shape(x)[0]
        y = self.norm(x)
        # global summary across tokens to get channel descriptors per sample
        # shape: (B, C)
        c_desc = tf.reduce_mean(y, axis=1)
        # bottleneck and expand
        z = self.proj1(c_desc)         # (B, C/reduction)
        z = self.drop(z, training=training)
        z = self.proj2(z)              # (B, C)
        # create per-channel gating weights via sigmoid
        gates = tf.nn.sigmoid(z)       # (B, C)
        gates = tf.reshape(gates, (B, 1, self.C))  # (B,1,C)
        out = x * gates                # broadcast multiply across tokens
        return x + out                 # residual

class GatedMLP(layers.Layer):
    """Gated MLP: expand -> gelu -> project -> gated by sigmoid branch -> residual"""
    def __init__(self, hidden_ratio=4, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.hidden_ratio = hidden_ratio
        self.dropout = dropout
    def build(self, input_shape):
        self.C = int(input_shape[-1])
        self.fc1 = layers.Dense(int(self.C * self.hidden_ratio), activation='gelu')
        self.fc2 = layers.Dense(self.C)
        # gating branch
        self.gate_fc = layers.Dense(self.C, activation='sigmoid')
        self.norm = layers.LayerNormalization(epsilon=1e-6)
        self.drop = layers.Dropout(self.dropout)
        super().build(input_shape)
    def call(self, x, training=False):
        # x: (B,N,C)
        y = self.norm(x)
        h = self.fc1(y)           # (B,N, C*ratio)
        h = self.drop(h, training=training)
        h = self.fc2(h)           # (B,N,C)
        gate = self.gate_fc(y)    # (B,N,C) sigmoid gating
        h = h * gate
        h = self.drop(h, training=training)
        return x + h              # residual

# ----------------------------
# NeuroFusion Block: stack modules
# ----------------------------
class NeuroFusionBlock(layers.Layer):
    def __init__(self, dim, token_kernel=3, groups=4, mlp_ratio=4, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.token_kernel = token_kernel
        self.groups = groups
        self.mlp_ratio = mlp_ratio
        self.dropout = dropout
        # sublayers
        self.ltm = LocalTokenMixer(kernel_size=self.token_kernel, dropout=self.dropout)
        self.gca = GroupedChannelAttention(groups=self.groups, reduction=max(1, self.mlp_ratio), dropout=self.dropout)
        self.gmlp = GatedMLP(hidden_ratio=self.mlp_ratio, dropout=self.dropout)
    def call(self, x, training=False):
        x = self.ltm(x, training=training)
        x = self.gca(x, training=training)
        x = self.gmlp(x, training=training)
        return x

# ----------------------------
# Hybrid: InceptionV3 + NeuroFusion head
# ----------------------------
def build_Inception_NeuroFusion(input_shape=(224,224,3),
                          patch_size=(8,8),
                          proj_dim=128,
                          num_novel_blocks=2,
                          token_kernel=3,
                          groups=4,
                          mlp_ratio=4,
                          classifier_hidden=128,
                          num_classes=1,
                          freeze_backbone=True):
    """
    Keep data flow:
      Input -> InceptionV3 (no top) -> 1x1 proj -> PatchExtractor -> NeuroFusionBlocks -> readout -> classifier
    """
    inputs = Input(shape=input_shape, name="input")
    base = InceptionV3(include_top=False, weights='imagenet', input_tensor=inputs)
    if freeze_backbone:
        base.trainable = False
    feats = base.output                                # (B,Hf,Wf,Cb)
    feats = layers.Conv2D(proj_dim, 1, activation='relu', name='proj')(feats)
    feats = layers.LayerNormalization(name='feat_norm')(feats)

    # produce tokens: (B, N, proj_dim)
    patches = PatchExtractor(patch_size=patch_size, name='patch_ex')(feats)

    x = patches  # tokens
    # stack NeuroFusion
    for i in range(num_novel_blocks):
        x = NeuroFusionBlock(dim=proj_dim, token_kernel=token_kernel, groups=groups,
                       mlp_ratio=mlp_ratio, dropout=0.1, name=f'novel_block_{i}')(x)

    # readout & classification head
    x = layers.GlobalAveragePooling1D(name='readout')(x)
    x = layers.Dense(classifier_hidden, activation='relu', name='clf')(x)
    x = layers.Dropout(0.1)(x)
    out = layers.Dense(1, activation='sigmoid' if num_classes==1 else 'softmax', name='out')(x)
    return Model(inputs, out, name='Inception_NeuroFusionHead')


if __name__ == "__main__":
    model = build_Inception_NeuroFusion(input_shape=(224,224,3),
                                  patch_size=(8,8),
                                  proj_dim=128,
                                  num_novel_blocks=3,
                                  token_kernel=3,
                                  groups=8,
                                  mlp_ratio=4,
                                  classifier_hidden=128,
                                  num_classes=1,
                                  freeze_backbone=True)
    model.summary()


In [ ]:
# Compile model
model.compile(optimizer=Adam(learning_rate=0.00001), loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
from tensorflow.keras.utils import plot_model
plot_model(model, to_file='model.png', show_shapes=False, show_layer_names=True, expand_nested=True)

In [ ]:
class CustomEarlyStopping(Callback):
    def __init__(self, target_accuracy):
        super(CustomEarlyStopping, self).__init__()
        self.target_accuracy = target_accuracy

    def on_epoch_end(self, epoch, logs=None):
        # Check if the validation accuracy has reached the target accuracy
        val_accuracy = logs.get('val_accuracy')
        if val_accuracy is not None and val_accuracy >= self.target_accuracy:
            self.model.stop_training = True

target_val_accuracy = 0.98
custom_early_stopping = CustomEarlyStopping(target_accuracy=target_val_accuracy)

In [ ]:
# Train model
epochs = 100
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epochs, batch_size=32, callbacks=[custom_early_stopping])

In [ ]:
# Plot the training and validation loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
# Plot the training and validation accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test)
y_pred_binary = (y_pred > 0.5).astype(int)  # Convert probabilities to binary predictions (0 or 1)

In [ ]:
# Evaluate the model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

accuracy = accuracy_score(y_test, y_pred_binary)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)
conf_matrix = confusion_matrix(y_test, y_pred_binary)

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")
print(f"Confusion Matrix:\n{conf_matrix}")

In [ ]:
from sklearn.metrics import classification_report

# Assuming y_test and y_pred_binary from the previous response
report = classification_report(y_test, y_pred_binary, target_names=['Healthy', 'DME'])
print(report)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Assuming you have y_test and y_pred_binary from the previous responses
cm = confusion_matrix(y_test, y_pred_binary)

# Create a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Healthy', 'DME'], yticklabels=['Healthy', 'DME'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

# Assuming you have y_test and y_pred from previous responses
fpr, tpr, thresholds = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='Hybrid Inception-NeuroFusion Model' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Predictions')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load the input image
image_path = '/content/OCT_Dataset2/DME/0002.jpg'
img = cv2.imread(image_path)
img_display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert BGR to RGB for display
plt.imshow(img_display)  # Display the original image
# Preprocess the image
img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
(b, g, r) = cv2.split(img)
img = cv2.merge([r, g, b])  # Convert to RGB color format
img = img.astype('float32') / 255.0  # Normalize pixel values

# Make prediction
img = np.expand_dims(img, axis=0)  # Add batch dimension
prediction = model.predict(img)

# Interpret prediction
class_label = (prediction > 0.5).astype(int)[0][0]  # Get predicted class label (0 or 1)
class_names = ['Healthy', 'DME']  # Define class names
predicted_class = class_names[class_label]

# Display the image and prediction
plt.title(f"Predicted class: {predicted_class}")
plt.axis('off')  # Hide axes
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load the input image
image_path = '/content/OCT_Dataset2/Healthy/0002.jpg'
img = cv2.imread(image_path)
img_display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert BGR to RGB for display
plt.imshow(img_display)  # Display the original image
# Preprocess the image
img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
(b, g, r) = cv2.split(img)
img = cv2.merge([r, g, b])  # Convert to RGB color format
img = img.astype('float32') / 255.0  # Normalize pixel values

# Make prediction
img = np.expand_dims(img, axis=0)  # Add batch dimension
prediction = model.predict(img)

# Interpret prediction
class_label = (prediction > 0.5).astype(int)[0][0]  # Get predicted class label (0 or 1)
class_names = ['Healthy', 'DME']  # Define class names
predicted_class = class_names[class_label]

# Display the image and prediction
plt.title(f"Predicted class: {predicted_class}")
plt.axis('off')  # Hide axes
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get a random sample of 15 images and their corresponding labels
num_images_to_display = 15
random_indices = np.random.choice(len(X_test), num_images_to_display, replace=False)
sample_images = X_test[random_indices]
sample_labels = y_test[random_indices]
sample_predictions = y_pred_binary[random_indices]  # Assuming y_pred_binary is available

# Create a figure and axes for the images
fig, axes = plt.subplots(3, 5, figsize=(15, 9))  # Adjust figsize as needed
axes = axes.flatten()  # Flatten the axes array for easier iteration

# Display each image with its label
for i, ax in enumerate(axes):
    ax.imshow(sample_images[i])
    ax.set_title(f"True: {class_names[int(sample_labels[i])]}\nPredicted: {class_names[int(sample_predictions[i])]}")
    ax.axis('off')  # Hide axes ticks

plt.tight_layout()
plt.show()